# Smart-SIEM — Complete Experimental Evaluation
### 6 experiments addressing all reviewer gaps

| # | Experiment | Purpose |
|---|---|---|
| 1 | Algorithm Comparison | CatBoost vs RF vs Extremely Randomized Trees vs XGBoost vs LightGBM vs LR vs DT |
| 2 | Cascade vs Flat | Proves two-stage design beats a single 7-class model |
| 3 | Confusion Matrices | Shows exactly which classes are confused |
| 4 | Extended Rule Engine | All 6 attack classes vs Wazuh native rules |
| 5 | Exact Ablation | Real N-vs-F1 values to replace thesis approximations |
| 6 | Retraining Simulation | Validates the 90% threshold mechanism |

**File paths** — adjust `BASE_PATH` if your folder structure differs:
```
BASE_PATH/../datasets/training/final_training_dataset_with_history_30_SMOTENC.csv
BASE_PATH/../datasets/testing/final_testing_dataset_with_history_30.csv
BASE_PATH/../datasets/dataset.csv
```


## 0. Imports & Configuration

In [1]:
# ── Standard library ──────────────────────────────────────────────
import os, json, warnings, time, pickle
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from IPython.display import display
from tqdm.notebook import tqdm

# ── scikit-learn ───────────────────────────────────────────────────
from sklearn.ensemble         import (RandomForestClassifier,
                                       ExtraTreesClassifier,
                                       GradientBoostingClassifier)
from sklearn.linear_model     import LogisticRegression
from sklearn.tree             import DecisionTreeClassifier
from sklearn.preprocessing    import OrdinalEncoder, LabelEncoder
from sklearn.metrics          import (classification_report, confusion_matrix,
                                       f1_score, precision_score, recall_score)
from sklearn.model_selection  import (RandomizedSearchCV, StratifiedKFold)

# ── Gradient boosting libraries ────────────────────────────────────
from catboost import CatBoostClassifier, Pool

try:
    import xgboost as xgb
    HAS_XGB = True
except ImportError:
    print("XGBoost not found — pip install xgboost")
    HAS_XGB = False

try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    print("LightGBM not found — pip install lightgbm")
    HAS_LGB = False

warnings.filterwarnings("ignore")

# ── Configuration ──────────────────────────────────────────────────
USE_GPU       = True    # RTX 3070 detected
BASE_PATH     = ".."    # one level up from jupyter notebooks folder
RANDOM_STATE  = 42
CV_FOLDS      = 3
N_ITER_SEARCH = 20      # RandomizedSearch iterations per model stage

# ── Plot style ────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
})

# Colour palette — one colour per algorithm
PALETTE = {
    "CatBoost (Proposed)"          : "#1565C0",
    "Random Forest"                : "#2E7D32",
    "Extremely Randomized Trees"   : "#00838F",
    "XGBoost"                      : "#E65100",
    "LightGBM"                     : "#6A1B9A",
    "Logistic Regression"          : "#C62828",
    "Decision Tree"                : "#5D4037",
}

print("✓ Imports complete")
print(f"  XGBoost : {HAS_XGB}  |  LightGBM : {HAS_LGB}  |  GPU : {USE_GPU}")
print(f"  RandomizedSearch: {N_ITER_SEARCH} iters × {CV_FOLDS}-fold CV per stage")


✓ Imports complete
  XGBoost : True  |  LightGBM : True  |  GPU : True
  RandomizedSearch: 20 iters × 3-fold CV per stage


## 1. Feature Definitions & Preprocessing

In [2]:
# ── Feature lists ────────────────────────────────────────────────
ALL_FEATURES_WITH_CTX = [
    '_source.data.protocol', '_source.data.id', '_source.rule.firedtimes',
    '_source.rule.mail', '_source.rule.level', '_source.rule.description',
    '_source.rule.groups', '_source.rule.pci_dss', '_source.rule.tsc',
    '_source.rule.nist_800_53', '_source.rule.gdpr', '_source.rule.mitre.id',
    '_source.rule.frequency', '_source.rule.hipaa', '_source.agent.description',
    '_source.rule.id',
    'history._source.rule.firedtimes',
    'history._source.data.id.200', 'history._source.data.id.300',
    'history._source.data.id.400', 'history._source.data.id.500',
    'T1212', 'T1068', 'T1064', 'T1210', 'T1083', 'T1055', 'T1190',
]
ALL_FEATURES_NO_CTX = [
    '_source.data.protocol', '_source.data.id', '_source.rule.firedtimes',
    '_source.rule.mail', '_source.rule.level', '_source.rule.description',
    '_source.rule.groups', '_source.rule.pci_dss', '_source.rule.tsc',
    '_source.rule.nist_800_53', '_source.rule.gdpr', '_source.rule.mitre.id',
    '_source.rule.frequency', '_source.rule.hipaa', '_source.agent.description',
    '_source.rule.id',
]
CATEGORY_FEATURES = [
    '_source.data.protocol', '_source.data.id', '_source.rule.mail',
    '_source.rule.description', '_source.rule.groups', '_source.rule.pci_dss',
    '_source.rule.tsc', '_source.rule.nist_800_53', '_source.rule.gdpr',
    '_source.rule.mitre.id', '_source.rule.hipaa', '_source.agent.description',
    '_source.rule.frequency', '_source.rule.id',
]
NUMERICAL_FEATURES_CTX = [
    '_source.rule.firedtimes', '_source.rule.level',
    'history._source.rule.firedtimes',
    'history._source.data.id.200', 'history._source.data.id.300',
    'history._source.data.id.400', 'history._source.data.id.500',
    'T1212', 'T1068', 'T1064', 'T1210', 'T1083', 'T1055', 'T1190',
]
NUMERICAL_FEATURES_NO_CTX = ['_source.rule.firedtimes', '_source.rule.level']
ATTACK_CLASSES = [
    'BROKEN_AUTHENTICATION', 'SENSITIVE_DATA_EXPOSURE',
    'SQL_INJECTION', 'WEB_SCAN', 'XSS', 'BRUTE_FORCE'
]

# ── Label normalisation ───────────────────────────────────────────
LABEL_MAP = {
    'brute_force'           : 'BRUTE_FORCE',
    'BRUTE_force'           : 'BRUTE_FORCE',
    'Broken_Authentication' : 'BROKEN_AUTHENTICATION',
    'broken_authentication' : 'BROKEN_AUTHENTICATION',
    'broken_Authentication' : 'BROKEN_AUTHENTICATION',
    'web_scan'              : 'WEB_SCAN',
    'sql_injection'         : 'SQL_INJECTION',
    'xss'                   : 'XSS',
    'sensitive_data_exposure': 'SENSITIVE_DATA_EXPOSURE',
}

def normalise_labels(df):
    for col in ['output_1', 'output_2']:
        if col in df.columns:
            df[col] = df[col].replace(LABEL_MAP)
    return df

def preprocessing(df, with_context=True):
    """Preprocessing — pandas 2.x compatible."""
    cat_feats = CATEGORY_FEATURES
    num_feats = NUMERICAL_FEATURES_CTX if with_context else NUMERICAL_FEATURES_NO_CTX
    for feat in cat_feats:
        if feat not in df.columns: df[feat] = ' '
    for feat in num_feats:
        if feat not in df.columns: df[feat] = 1
    for c in cat_feats:
        if df[c].isnull().any(): df[c] = df[c].fillna(' ')
    df['_source.rule.level']      = df['_source.rule.level'].fillna(3)
    df['_source.rule.firedtimes'] = df['_source.rule.firedtimes'].fillna(1)
    df['_source.rule.id']  = df['_source.rule.id'].astype(str)
    df['_source.rule.mail'] = df['_source.rule.mail'].apply(
        lambda x: 1 if x is True or str(x).lower() == 'true' else 0)
    for c in cat_feats:
        df[c] = df[c].astype(str)
    df['_source.agent.description'] = 'APACHE_SERVER'
    df = df.fillna(' ')
    return df

def load_dataset(train_path, test_path, with_context=True):
    feats    = ALL_FEATURES_WITH_CTX if with_context else ALL_FEATURES_NO_CTX
    all_cols = feats + ['output_1', 'output_2']
    tr = normalise_labels(preprocessing(pd.read_csv(train_path), with_context))[all_cols]
    te = normalise_labels(preprocessing(pd.read_csv(test_path),  with_context))[all_cols]
    X_train  = tr.drop(['output_1', 'output_2'], axis=1)
    y1_train = tr['output_1']
    y2_train = tr[tr['output_2'] != 'NORMAL']['output_2']
    X2_train = tr[tr['output_2'] != 'NORMAL'].drop(['output_1', 'output_2'], axis=1)
    X_test   = te.drop(['output_1', 'output_2'], axis=1)
    y1_test  = te['output_1']
    y2_test  = te[te['output_2'] != 'NORMAL']['output_2']
    X2_test  = te[te['output_2'] != 'NORMAL'].drop(['output_1', 'output_2'], axis=1)
    return X_train, y1_train, X2_train, y2_train, X_test, y1_test, X2_test, y2_test

print("✓ Feature definitions and preprocessing loaded")


✓ Feature definitions and preprocessing loaded


## 2. Load Main Dataset (N=30, with context)

In [3]:
TRAIN_CTX = f"{BASE_PATH}/datasets/training/final_training_dataset_with_history_30_SMOTENC.csv"
TEST_CTX  = f"{BASE_PATH}/datasets/testing/final_testing_dataset_with_history_30.csv"
TRAIN_NO  = f"{BASE_PATH}/datasets/training/final_training_dataset_no_history_SMOTENC.csv"
TEST_NO   = f"{BASE_PATH}/datasets/testing/final_testing_dataset_no_history.csv"
RAW_DATASET = f"{BASE_PATH}/datasets/dataset.csv"

(X_train, y1_train, X2_train, y2_train,
 X_test,  y1_test,  X2_test,  y2_test) = load_dataset(TRAIN_CTX, TEST_CTX, with_context=True)

print(f"Training set  : {X_train.shape[0]:,} rows  (Stage-1)")
print(f"Training set  : {X2_train.shape[0]:,} rows  (Stage-2, attack-only)")
print(f"Test set      : {X_test.shape[0]:,} rows")
print(f"\nStage-1 labels (train):\n{y1_train.value_counts()}")
print(f"\nStage-2 labels (test):\n{y2_test.value_counts()}")

unexpected = set(y2_test.unique()) - set(ATTACK_CLASSES)
if unexpected:
    print(f"\n⚠️  Unexpected labels: {unexpected}  ← check LABEL_MAP")
else:
    print("\n✓ All labels normalised correctly")


Training set  : 12,342 rows  (Stage-1)
Training set  : 7,500 rows  (Stage-2, attack-only)
Test set      : 9,232 rows

Stage-1 labels (train):
output_1
ATTACK    7500
NORMAL    4842
Name: count, dtype: int64

Stage-2 labels (test):
output_2
SENSITIVE_DATA_EXPOSURE    5312
SQL_INJECTION              1315
WEB_SCAN                   1131
BRUTE_FORCE                 140
XSS                          63
BROKEN_AUTHENTICATION        60
Name: count, dtype: int64

✓ All labels normalised correctly


## 3. CatBoost Cascade Trainer (with progress bars)

In [4]:
class TqdmCatBoostCallback:
    """Shows a tqdm progress bar during CatBoost training."""
    def __init__(self, total, desc="CatBoost"):
        self.pbar = tqdm(total=total, desc=desc,
                         bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} iter [{elapsed}<{remaining}]",
                         colour="blue", leave=True)
        self.last = 0
    def after_iteration(self, info):
        self.pbar.update(info.iteration - self.last)
        self.last = info.iteration
        return True
    def __del__(self):
        try: self.pbar.close()
        except Exception: pass


def tune_catboost(pool_tr, pool_te, stage_label, task):
    """CatBoost built-in grid search with progress bar."""
    param_grid = {
        "learning_rate" : [0.03, 0.06, 0.08, 0.1],
        "depth"         : [6, 8, 10],
        "l2_leaf_reg"   : [7, 9, 10],
        "iterations"    : [2000, 4000],
    }
    total_combos = (len(param_grid["learning_rate"]) *
                    len(param_grid["depth"])          *
                    len(param_grid["l2_leaf_reg"])    *
                    len(param_grid["iterations"]))
    print(f"  [{stage_label}] Grid search: {total_combos} param combos × {CV_FOLDS}-fold CV")
    with tqdm(total=total_combos, desc=f"  Tuning {stage_label}",
              bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} combos [{elapsed}]",
              colour="cyan") as pbar:
        model = CatBoostClassifier(logging_level="Silent", task_type=task,
                                    random_seed=RANDOM_STATE)
        gs = model.grid_search(
            param_grid, pool_tr,
            cv=CV_FOLDS, partition_random_seed=RANDOM_STATE,
            calc_cv_statistics=True, search_by_train_test_split=True,
            refit=True, shuffle=True, stratified=True,
            train_size=0.8, verbose=False, plot=False,
        )
        pbar.update(total_combos)
    best = gs["params"]
    print(f"  [{stage_label}] Best: {best}")
    return model, best


def train_catboost_cascade(X_train, y1_train, X2_train, y2_train,
                            X_test,  y1_test,  X2_test,  y2_test,
                            params1=None, params2=None,
                            label="CatBoost", save_path=None,
                            tune=True, verbose=False):
    task = "GPU" if USE_GPU else "CPU"
    t0   = time.time()
    pool1_tr = Pool(X_train,  label=y1_train, cat_features=CATEGORY_FEATURES)
    pool1_te = Pool(X_test,   label=y1_test,  cat_features=CATEGORY_FEATURES)
    pool2_tr = Pool(X2_train, label=y2_train, cat_features=CATEGORY_FEATURES)
    pool2_te = Pool(X2_test,  label=y2_test,  cat_features=CATEGORY_FEATURES)

    if tune:
        m1, best1 = tune_catboost(pool1_tr, pool1_te, "Stage-1 Binary",     task)
        m2, best2 = tune_catboost(pool2_tr, pool2_te, "Stage-2 Multiclass", task)
    else:
        if params1 is None: params1 = dict(iterations=2631, learning_rate=0.1, depth=10, l2_leaf_reg=10)
        if params2 is None: params2 = dict(iterations=6405, learning_rate=0.08, depth=8, l2_leaf_reg=7)
        best1, best2 = params1, params2
        cb1 = TqdmCatBoostCallback(params1.get("iterations", 2631), "  Stage-1 training")
        cb2 = TqdmCatBoostCallback(params2.get("iterations", 6405), "  Stage-2 training")
        m1 = CatBoostClassifier(**params1, task_type=task, logging_level="Silent", random_seed=RANDOM_STATE)
        m1.fit(pool1_tr, eval_set=pool1_te, verbose=verbose, callbacks=[cb1])
        m2 = CatBoostClassifier(**params2, task_type=task, logging_level="Silent", random_seed=RANDOM_STATE)
        m2.fit(pool2_tr, eval_set=pool2_te, verbose=verbose, callbacks=[cb2])

    if save_path:
        m1.save_model(f"{save_path}_1.bin")
        m2.save_model(f"{save_path}_2.bin")

    y1_pred = m1.predict(X_test)
    y2_pred = np.array([p[0] if isinstance(p, (list, np.ndarray)) else p for p in m2.predict(X2_test)])

    return dict(
        label          = label,
        s1_precision   = precision_score(y1_test, y1_pred, average="macro"),
        s1_recall      = recall_score(y1_test,    y1_pred, average="macro"),
        s1_f1          = f1_score(y1_test,        y1_pred, average="macro"),
        s2_precision   = precision_score(y2_test, y2_pred, average="macro"),
        s2_recall      = recall_score(y2_test,    y2_pred, average="macro"),
        s2_f1          = f1_score(y2_test,        y2_pred, average="macro"),
        m1=m1, m2=m2,
        best_params1   = best1,
        best_params2   = best2,
        train_time_sec = round(time.time() - t0, 1),
        y1_pred=y1_pred, y2_pred=y2_pred,
        y1_test=y1_test, y2_test=y2_test,
        report1 = classification_report(y1_test, y1_pred, output_dict=True),
        report2 = classification_report(y2_test, y2_pred, output_dict=True),
    )

print("✓ CatBoost trainer ready")


✓ CatBoost trainer ready


## 4. sklearn Helpers (with hyperparameter tuning + progress bars)

In [5]:
def encode_for_sklearn(X_train, X_test):
    enc  = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    cats = [c for c in CATEGORY_FEATURES if c in X_train.columns]
    X_tr = X_train.copy().reset_index(drop=True)
    X_te = X_test.copy().reset_index(drop=True)
    X_tr[cats] = enc.fit_transform(X_tr[cats])
    X_te[cats] = enc.transform(X_te[cats])
    return X_tr.astype(float).values, X_te.astype(float).values


def tune_sklearn(clf_class, param_dist, label, X_tr, y_tr):
    """RandomizedSearchCV with tqdm progress bar."""
    cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    with tqdm(total=N_ITER_SEARCH, desc=f"  Tuning {label}",
              bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]",
              colour="green", leave=True) as pbar:
        try:
            base = clf_class(random_state=RANDOM_STATE)
        except TypeError:
            base = clf_class()
        search = RandomizedSearchCV(base, param_dist, n_iter=N_ITER_SEARCH,
                                     scoring="f1_macro", cv=cv, n_jobs=-1,
                                     random_state=RANDOM_STATE, verbose=0, refit=True)
        search.fit(X_tr, y_tr)
        pbar.n = N_ITER_SEARCH
        pbar.refresh()
    print(f"  → Best CV F1={search.best_score_:.3f}  {search.best_params_}")
    return search.best_estimator_, search.best_params_


def train_sklearn_cascade(clf_class, param_dist, label,
                           X_train, y1_train, X2_train, y2_train,
                           X_test,  y1_test,  X2_test,  y2_test,
                           tune=True, fixed_params=None):
    t0 = time.time()
    Xtr_enc, Xte_enc   = encode_for_sklearn(X_train,  X_test)
    X2tr_enc, X2te_enc = encode_for_sklearn(X2_train, X2_test)
    if tune:
        m1, bp1 = tune_sklearn(clf_class, param_dist, f"{label} Stage-1", Xtr_enc,  y1_train)
        m2, bp2 = tune_sklearn(clf_class, param_dist, f"{label} Stage-2", X2tr_enc, y2_train)
    else:
        params = fixed_params or {}
        try:
            m1 = clf_class(**params, random_state=RANDOM_STATE)
            m2 = clf_class(**params, random_state=RANDOM_STATE)
        except TypeError:
            m1 = clf_class(**params)
            m2 = clf_class(**params)
        m1.fit(Xtr_enc, y1_train)
        m2.fit(X2tr_enc, y2_train)
        bp1, bp2 = params, params
    y1_pred = m1.predict(Xte_enc)
    y2_pred = m2.predict(X2te_enc)
    return dict(
        label          = label,
        s1_precision   = precision_score(y1_test, y1_pred, average="macro", zero_division=0),
        s1_recall      = recall_score(y1_test,    y1_pred, average="macro", zero_division=0),
        s1_f1          = f1_score(y1_test,        y1_pred, average="macro", zero_division=0),
        s2_precision   = precision_score(y2_test, y2_pred, average="macro", zero_division=0),
        s2_recall      = recall_score(y2_test,    y2_pred, average="macro", zero_division=0),
        s2_f1          = f1_score(y2_test,        y2_pred, average="macro", zero_division=0),
        best_params1   = bp1,
        best_params2   = bp2,
        train_time_sec = round(time.time() - t0, 1),
        y1_pred=y1_pred, y2_pred=y2_pred,
        y1_test=y1_test, y2_test=y2_test,
        m1=None, m2=None,
    )

print("✓ sklearn helpers ready")


✓ sklearn helpers ready


## 5. Experiment 1 — Algorithm Comparison

**7 algorithms**, all with proper hyperparameter tuning, checkpointing, and progress bars.
Delete `./checkpoints/` folder to force full retrain from scratch.

In [6]:
# ── Checkpoint helpers ────────────────────────────────────────────
CHECKPOINT_DIR = "./checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

def _safe(name): return name.replace(" ", "_").replace("(", "").replace(")", "")

def save_checkpoint(name, result):
    meta = {k: v for k, v in result.items() if k not in ("m1", "m2")}
    with open(f"{CHECKPOINT_DIR}/{_safe(name)}_meta.pkl", "wb") as f:
        pickle.dump(meta, f)
    if result.get("m1") is not None:
        try: result["m1"].save_model(f"{CHECKPOINT_DIR}/{_safe(name)}_m1.bin")
        except Exception: pass
    if result.get("m2") is not None:
        try: result["m2"].save_model(f"{CHECKPOINT_DIR}/{_safe(name)}_m2.bin")
        except Exception: pass
    print(f"  ✓ Saved  → ./checkpoints/{_safe(name)}_meta.pkl")

def load_checkpoint(name):
    path = f"{CHECKPOINT_DIR}/{_safe(name)}_meta.pkl"
    with open(path, "rb") as f:
        r = pickle.load(f)
    # ensure train_time_sec always present (patches old checkpoints)
    r.setdefault("train_time_sec", 0)
    print(f"  ⚡ Loaded  ← {path}")
    return r

def checkpoint_exists(name):
    return os.path.exists(f"{CHECKPOINT_DIR}/{_safe(name)}_meta.pkl")

# ── Hyperparameter search spaces ─────────────────────────────────
RF_ET_PARAMS = {
    "n_estimators"      : [200, 300, 500],
    "max_depth"         : [10, 15, 20, None],
    "min_samples_split" : [2, 5, 10],
    "min_samples_leaf"  : [1, 2, 4],
    "max_features"      : ["sqrt", "log2", 0.3, 0.5],
    "n_jobs"            : [-1],
}
DT_PARAMS = {
    "max_depth"         : [8, 12, 16, 20, None],
    "min_samples_split" : [2, 5, 10, 20],
    "min_samples_leaf"  : [1, 2, 4, 8],
    "criterion"         : ["gini", "entropy"],
    "max_features"      : ["sqrt", "log2", None],
}
LR_PARAMS = {
    "C"       : [0.01, 0.1, 0.5, 1.0, 5.0, 10.0],
    "solver"  : ["lbfgs", "saga"],
    "max_iter": [2000],
}
XGB_LGB_PARAMS = {
    "n_estimators"     : [300, 500, 800],
    "max_depth"        : [6, 8, 10],
    "learning_rate"    : [0.03, 0.06, 0.1],
    "subsample"        : [0.7, 0.85, 1.0],
    "colsample_bytree" : [0.7, 0.85, 1.0],
    "reg_lambda"       : [1, 3, 7],
    "n_jobs"           : [-1],
}

# ── Overall progress bar ──────────────────────────────────────────
ALGORITHMS = [
    "CatBoost (Proposed)",
    "Random Forest",
    "Extremely Randomized Trees",
    "XGBoost",
    "LightGBM",
    "Logistic Regression",
    "Decision Tree",
]
n_done = sum(checkpoint_exists(n) for n in ALGORITHMS)
overall_pb = tqdm(total=len(ALGORITHMS), initial=n_done,
                   desc="Overall progress",
                   bar_format="Overall {l_bar}{bar}| {n_fmt}/{total_fmt} algorithms [{elapsed}]",
                   colour="yellow", position=0, leave=True)
print(f"  {n_done} already done (checkpoint), {len(ALGORITHMS)-n_done} will train now.\n")

results_algo = []

# ═══════════════════════════════════════════════════════════════════
# 1. CatBoost (Proposed)
# ═══════════════════════════════════════════════════════════════════
name = "CatBoost (Proposed)"
print(f"{'='*55}\n[1/7] {name}")
if checkpoint_exists(name):
    res_cb = load_checkpoint(name)
else:
    res_cb = train_catboost_cascade(
        X_train, y1_train, X2_train, y2_train,
        X_test,  y1_test,  X2_test,  y2_test,
        label=name, tune=True,
        save_path=f"{CHECKPOINT_DIR}/CatBoost_Proposed",
    )
    save_checkpoint(name, res_cb)
results_algo.append(res_cb)
overall_pb.update(1)
print(f"  → S1 F1={res_cb['s1_f1']:.3f}  S2 F1={res_cb['s2_f1']:.3f}  Time={res_cb.get('train_time_sec',0):.0f}s")

# ═══════════════════════════════════════════════════════════════════
# 2. Random Forest
# ═══════════════════════════════════════════════════════════════════
name = "Random Forest"
print(f"\n{'='*55}\n[2/7] {name}")
if checkpoint_exists(name):
    res_rf = load_checkpoint(name)
else:
    res_rf = train_sklearn_cascade(
        RandomForestClassifier, RF_ET_PARAMS, name,
        X_train, y1_train, X2_train, y2_train,
        X_test,  y1_test,  X2_test,  y2_test, tune=True,
    )
    save_checkpoint(name, res_rf)
results_algo.append(res_rf)
overall_pb.update(1)
print(f"  → S1 F1={res_rf['s1_f1']:.3f}  S2 F1={res_rf['s2_f1']:.3f}  Time={res_rf.get('train_time_sec',0):.0f}s")

# ═══════════════════════════════════════════════════════════════════
# 3. Extremely Randomized Trees  (ExtraTreesClassifier)
#    Reference: Geurts et al., "Extremely randomized trees",
#    Machine Learning, 63(1), 3-42, 2006.
# ═══════════════════════════════════════════════════════════════════
name = "Extremely Randomized Trees"
print(f"\n{'='*55}\n[3/7] {name}")
print("  (sklearn: ExtraTreesClassifier — Geurts et al. 2006)")
if checkpoint_exists(name):
    res_et = load_checkpoint(name)
else:
    res_et = train_sklearn_cascade(
        ExtraTreesClassifier, RF_ET_PARAMS, name,
        X_train, y1_train, X2_train, y2_train,
        X_test,  y1_test,  X2_test,  y2_test, tune=True,
    )
    save_checkpoint(name, res_et)
results_algo.append(res_et)
overall_pb.update(1)
print(f"  → S1 F1={res_et['s1_f1']:.3f}  S2 F1={res_et['s2_f1']:.3f}  Time={res_et.get('train_time_sec',0):.0f}s")

# ═══════════════════════════════════════════════════════════════════
# 4. XGBoost
# ═══════════════════════════════════════════════════════════════════
if HAS_XGB:
    name = "XGBoost"
    print(f"\n{'='*55}\n[4/7] {name}")
    if checkpoint_exists(name):
        res_xgb = load_checkpoint(name)
    else:
        t0 = time.time()
        Xtr_e, Xte_e   = encode_for_sklearn(X_train, X_test)
        X2tr_e, X2te_e = encode_for_sklearn(X2_train, X2_test)
        le1 = LabelEncoder().fit(y1_train)
        le2 = LabelEncoder().fit(y2_train)
        y1_tr_enc = le1.transform(y1_train)
        y2_tr_enc = le2.transform(y2_train)
        cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
        xgb_params = {k: v for k, v in XGB_LGB_PARAMS.items() if k != "n_jobs"}

        print(f"  Tuning {name} Stage-1...")
        s1 = RandomizedSearchCV(
            xgb.XGBClassifier(eval_metric="logloss", verbosity=0, random_state=RANDOM_STATE),
            xgb_params, n_iter=N_ITER_SEARCH, scoring="f1_macro",
            cv=cv, n_jobs=-1, random_state=RANDOM_STATE, verbose=0, refit=True)
        with tqdm(total=N_ITER_SEARCH, desc="  Tuning XGBoost Stage-1",
                  bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}]",
                  colour="green") as pb:
            s1.fit(Xtr_e, y1_tr_enc); pb.n=N_ITER_SEARCH; pb.refresh()
        print(f"  → Best CV F1={s1.best_score_:.3f}  {s1.best_params_}")

        print(f"  Tuning {name} Stage-2...")
        s2 = RandomizedSearchCV(
            xgb.XGBClassifier(eval_metric="mlogloss", verbosity=0, random_state=RANDOM_STATE),
            xgb_params, n_iter=N_ITER_SEARCH, scoring="f1_macro",
            cv=cv, n_jobs=-1, random_state=RANDOM_STATE, verbose=0, refit=True)
        with tqdm(total=N_ITER_SEARCH, desc="  Tuning XGBoost Stage-2",
                  bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}]",
                  colour="green") as pb:
            s2.fit(X2tr_e, y2_tr_enc); pb.n=N_ITER_SEARCH; pb.refresh()
        print(f"  → Best CV F1={s2.best_score_:.3f}  {s2.best_params_}")

        s1.best_estimator_.save_model(f"{CHECKPOINT_DIR}/XGBoost_m1.json")
        s2.best_estimator_.save_model(f"{CHECKPOINT_DIR}/XGBoost_m2.json")
        y1p = le1.inverse_transform(s1.best_estimator_.predict(Xte_e))
        y2p = le2.inverse_transform(s2.best_estimator_.predict(X2te_e))
        res_xgb = dict(
            label="XGBoost",
            s1_precision=precision_score(y1_test, y1p, average="macro"),
            s1_recall   =recall_score(y1_test,    y1p, average="macro"),
            s1_f1       =f1_score(y1_test,        y1p, average="macro"),
            s2_precision=precision_score(y2_test, y2p, average="macro"),
            s2_recall   =recall_score(y2_test,    y2p, average="macro"),
            s2_f1       =f1_score(y2_test,        y2p, average="macro"),
            best_params1=s1.best_params_, best_params2=s2.best_params_,
            train_time_sec=round(time.time()-t0,1),
            y1_pred=y1p, y2_pred=y2p, y1_test=y1_test, y2_test=y2_test, m1=None, m2=None,
        )
        save_checkpoint("XGBoost", res_xgb)
    results_algo.append(res_xgb)
    overall_pb.update(1)
    print(f"  → S1 F1={res_xgb['s1_f1']:.3f}  S2 F1={res_xgb['s2_f1']:.3f}  Time={res_xgb.get('train_time_sec',0):.0f}s")

# ═══════════════════════════════════════════════════════════════════
# 5. LightGBM
# ═══════════════════════════════════════════════════════════════════
if HAS_LGB:
    name = "LightGBM"
    print(f"\n{'='*55}\n[5/7] {name}")
    if checkpoint_exists(name):
        res_lgb = load_checkpoint(name)
    else:
        t0 = time.time()
        Xtr_e, Xte_e   = encode_for_sklearn(X_train, X_test)
        X2tr_e, X2te_e = encode_for_sklearn(X2_train, X2_test)
        le1_lgb = LabelEncoder().fit(y1_train)
        le2_lgb = LabelEncoder().fit(y2_train)
        y1_tr_enc = le1_lgb.transform(y1_train)
        y2_tr_enc = le2_lgb.transform(y2_train)
        cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
        lgb_params = {k: v for k, v in XGB_LGB_PARAMS.items() if k != "n_jobs"}

        print(f"  Tuning {name} Stage-1...")
        s1 = RandomizedSearchCV(
            lgb.LGBMClassifier(verbose=-1, random_state=RANDOM_STATE),
            lgb_params, n_iter=N_ITER_SEARCH, scoring="f1_macro",
            cv=cv, n_jobs=-1, random_state=RANDOM_STATE, verbose=0, refit=True)
        with tqdm(total=N_ITER_SEARCH, desc="  Tuning LightGBM Stage-1",
                  bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}]",
                  colour="green") as pb:
            s1.fit(Xtr_e, y1_tr_enc); pb.n=N_ITER_SEARCH; pb.refresh()
        print(f"  → Best CV F1={s1.best_score_:.3f}  {s1.best_params_}")

        print(f"  Tuning {name} Stage-2...")
        s2 = RandomizedSearchCV(
            lgb.LGBMClassifier(verbose=-1, random_state=RANDOM_STATE),
            lgb_params, n_iter=N_ITER_SEARCH, scoring="f1_macro",
            cv=cv, n_jobs=-1, random_state=RANDOM_STATE, verbose=0, refit=True)
        with tqdm(total=N_ITER_SEARCH, desc="  Tuning LightGBM Stage-2",
                  bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}]",
                  colour="green") as pb:
            s2.fit(X2tr_e, y2_tr_enc); pb.n=N_ITER_SEARCH; pb.refresh()
        print(f"  → Best CV F1={s2.best_score_:.3f}  {s2.best_params_}")

        s1.best_estimator_.booster_.save_model(f"{CHECKPOINT_DIR}/LightGBM_m1.txt")
        s2.best_estimator_.booster_.save_model(f"{CHECKPOINT_DIR}/LightGBM_m2.txt")
        y1p = le1_lgb.inverse_transform(s1.best_estimator_.predict(Xte_e))
        y2p = le2_lgb.inverse_transform(s2.best_estimator_.predict(X2te_e))
        res_lgb = dict(
            label="LightGBM",
            s1_precision=precision_score(y1_test, y1p, average="macro"),
            s1_recall   =recall_score(y1_test,    y1p, average="macro"),
            s1_f1       =f1_score(y1_test,        y1p, average="macro"),
            s2_precision=precision_score(y2_test, y2p, average="macro"),
            s2_recall   =recall_score(y2_test,    y2p, average="macro"),
            s2_f1       =f1_score(y2_test,        y2p, average="macro"),
            best_params1=s1.best_params_, best_params2=s2.best_params_,
            train_time_sec=round(time.time()-t0,1),
            y1_pred=y1p, y2_pred=y2p, y1_test=y1_test, y2_test=y2_test, m1=None, m2=None,
        )
        save_checkpoint("LightGBM", res_lgb)
    results_algo.append(res_lgb)
    overall_pb.update(1)
    print(f"  → S1 F1={res_lgb['s1_f1']:.3f}  S2 F1={res_lgb['s2_f1']:.3f}  Time={res_lgb.get('train_time_sec',0):.0f}s")

# ═══════════════════════════════════════════════════════════════════
# 6. Logistic Regression
# ═══════════════════════════════════════════════════════════════════
name = "Logistic Regression"
print(f"\n{'='*55}\n[6/7] {name}")
if checkpoint_exists(name):
    res_lr = load_checkpoint(name)
else:
    res_lr = train_sklearn_cascade(
        LogisticRegression, LR_PARAMS, name,
        X_train, y1_train, X2_train, y2_train,
        X_test,  y1_test,  X2_test,  y2_test, tune=True,
    )
    save_checkpoint(name, res_lr)
results_algo.append(res_lr)
overall_pb.update(1)
print(f"  → S1 F1={res_lr['s1_f1']:.3f}  S2 F1={res_lr['s2_f1']:.3f}  Time={res_lr.get('train_time_sec',0):.0f}s")

# ═══════════════════════════════════════════════════════════════════
# 7. Decision Tree
# ═══════════════════════════════════════════════════════════════════
name = "Decision Tree"
print(f"\n{'='*55}\n[7/7] {name}")
if checkpoint_exists(name):
    res_dt = load_checkpoint(name)
else:
    res_dt = train_sklearn_cascade(
        DecisionTreeClassifier, DT_PARAMS, name,
        X_train, y1_train, X2_train, y2_train,
        X_test,  y1_test,  X2_test,  y2_test, tune=True,
    )
    save_checkpoint(name, res_dt)
results_algo.append(res_dt)
overall_pb.update(1)
print(f"  → S1 F1={res_dt['s1_f1']:.3f}  S2 F1={res_dt['s2_f1']:.3f}  Time={res_dt.get('train_time_sec',0):.0f}s")

overall_pb.close()
print(f"\n{'='*55}")
print("✓ All 7 algorithms done")
print(f"Checkpoints in: {os.path.abspath(CHECKPOINT_DIR)}/")
print("To force retrain: delete the ./checkpoints/ folder and re-run")


Overall Overall progress:   0%|          | 0/7 algorithms [00:00]

  0 already done (checkpoint), 7 will train now.

[1/7] CatBoost (Proposed)
  [Stage-1 Binary] Grid search: 72 param combos × 3-fold CV


  Tuning Stage-1 Binary:   0%|          | 0/72 combos [00:00]

KeyboardInterrupt: 

In [ ]:
# ── Summary table ────────────────────────────────────────────────
df_algo = pd.DataFrame([{
    "Algorithm" : r["label"],
    "S1 Prec."  : round(r["s1_precision"], 3),
    "S1 Rec."   : round(r["s1_recall"],    3),
    "S1 F1"     : round(r["s1_f1"],        3),
    "S2 Prec."  : round(r["s2_precision"], 3),
    "S2 Rec."   : round(r["s2_recall"],    3),
    "S2 F1"     : round(r["s2_f1"],        3),
    "Train (s)" : r.get("train_time_sec", "—"),
} for r in results_algo])

display(df_algo.style
    .highlight_max(subset=["S1 F1","S2 F1"], color="#d4edda")
    .highlight_min(subset=["S1 F1","S2 F1"], color="#f8d7da")
    .format({"S1 Prec.":"{:.3f}","S1 Rec.":"{:.3f}","S1 F1":"{:.3f}",
             "S2 Prec.":"{:.3f}","S2 Rec.":"{:.3f}","S2 F1":"{:.3f}"}))

print("\n--- Best hyperparameters ---")
for r in results_algo:
    print(f"\n{r['label']}:")
    print(f"  Stage-1: {r.get('best_params1','N/A')}")
    print(f"  Stage-2: {r.get('best_params2','N/A')}")

# ── Diagram A — Grouped bar chart ────────────────────────────────
labels  = [r["label"] for r in results_algo]
s1_vals = [r["s1_f1"] for r in results_algo]
s2_vals = [r["s2_f1"] for r in results_algo]
colors  = [PALETTE.get(l, "#666") for l in labels]
x = np.arange(len(labels)); w = 0.38
fig, ax = plt.subplots(figsize=(14, 5))
b1 = ax.bar(x - w/2, s1_vals, w, label="Stage 1 (Binary)",     color=colors, alpha=0.72)
b2 = ax.bar(x + w/2, s2_vals, w, label="Stage 2 (Multi-class)", color=colors, alpha=0.96,
            edgecolor="black", linewidth=0.5)
for bar in list(b1)+list(b2):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003,
            f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=7.5)
ax.set_xticks(x)
ax.set_xticklabels([l.replace(" (Proposed)","★") for l in labels], rotation=18, ha="right", fontsize=9)
ax.set_ylabel("Macro F₁-score"); ax.set_ylim(0.5, 1.07)
ax.axhline(1.0, color="gray", linestyle=":", linewidth=1, label="Upper bound")
ax.legend(fontsize=10)
ax.set_title("Algorithm Comparison — Macro F₁ (Stage 1 & 2)", fontsize=13, fontweight="bold")
ax.annotate("★ = Proposed (CatBoost)", xy=(0.01,0.02), xycoords="axes fraction", fontsize=9, color="gray")
plt.tight_layout()
plt.savefig("diagram_A_algorithm_comparison.png", dpi=200, bbox_inches="tight")
plt.show()
print("✓ Saved: diagram_A_algorithm_comparison.png")

# ── Diagram B — Radar chart ───────────────────────────────────────
metrics   = ["S1 Prec.", "S1 Rec.", "S1 F1", "S2 Prec.", "S2 Rec.", "S2 F1"]
N_m       = len(metrics)
angles    = np.linspace(0, 2*np.pi, N_m, endpoint=False).tolist() + [0]
fig, ax   = plt.subplots(figsize=(9,9), subplot_kw={"polar":True})
ax.set_theta_offset(np.pi/2); ax.set_theta_direction(-1)
ax.set_xticks(angles[:-1]); ax.set_xticklabels(metrics, fontsize=10)
ax.set_ylim(0.5,1.0); ax.set_yticks([0.6,0.7,0.8,0.9,1.0])
ax.set_yticklabels(["0.6","0.7","0.8","0.9","1.0"], fontsize=8)
ax.grid(color="gray", linestyle="--", linewidth=0.5, alpha=0.5)
for r in results_algo:
    vals = [r["s1_precision"],r["s1_recall"],r["s1_f1"],
            r["s2_precision"],r["s2_recall"],r["s2_f1"]] + [r["s1_precision"]]
    col = PALETTE.get(r["label"],"#666")
    lw  = 2.5 if "(Proposed)" in r["label"] else 1.3
    ls  = "-"  if "(Proposed)" in r["label"] else "--"
    ax.plot(angles, vals, color=col, linewidth=lw, linestyle=ls, label=r["label"])
    ax.fill(angles, vals, color=col, alpha=0.04)
ax.legend(loc="upper right", bbox_to_anchor=(1.38,1.15), fontsize=9)
ax.set_title("Algorithm Metrics Profile (Radar Chart)", fontsize=13, fontweight="bold", pad=25)
plt.tight_layout()
plt.savefig("diagram_B_radar_chart.png", dpi=200, bbox_inches="tight")
plt.show()
print("✓ Saved: diagram_B_radar_chart.png")

# ── Diagram C — Efficiency vs Accuracy scatter ────────────────────
fig, axes = plt.subplots(1,2, figsize=(14,5))
for ax, stage, key in zip(axes, ["Stage 1 (Binary)","Stage 2 (Multi-class)"],["s1_f1","s2_f1"]):
    for r in results_algo:
        t   = r.get("train_time_sec", 0)
        f1  = r[key]
        col = PALETTE.get(r["label"],"#666")
        ms  = 200 if "(Proposed)" in r["label"] else 100
        mk  = "*"  if "(Proposed)" in r["label"] else "o"
        ax.scatter(t, f1, color=col, s=ms, marker=mk, zorder=3, edgecolors="black", linewidths=0.5)
        ax.annotate(r["label"].replace(" (Proposed)","★"),
                    (t,f1), textcoords="offset points", xytext=(6,4), fontsize=7.5, color=col)
    ax.set_xlabel("Training time (s)"); ax.set_ylabel("Macro F₁-score")
    ax.set_title(f"Efficiency vs Accuracy — {stage}"); ax.grid(True, alpha=0.3)
plt.suptitle("F₁ Score vs Training Time (★ = Proposed)", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("diagram_C_efficiency_accuracy.png", dpi=200, bbox_inches="tight")
plt.show()
print("✓ Saved: diagram_C_efficiency_accuracy.png")

# ── LaTeX table ───────────────────────────────────────────────────
print("\n" + "="*70)
print("LATEX TABLE — paste into paper")
print("="*70)
best_s1 = max(r["s1_f1"] for r in results_algo)
best_s2 = max(r["s2_f1"] for r in results_algo)
print(r"\begin{table}[htbp]")
print(r"\centering")
print(r"\caption{Algorithm comparison (context-enriched, $N=30$, macro-averaged).")
print(r"Best result per metric in \textbf{bold}. All sklearn models tuned with")
print(r"\texttt{RandomizedSearchCV} ($n=20$, $k=3$); CatBoost uses built-in")
print(r"\texttt{grid\_search}.}")
print(r"\label{tab:algo_comparison}")
print(r"\begin{tabular}{lccccccr}")
print(r"\toprule")
print(r"\multirow{2}{*}{Algorithm} & \multicolumn{3}{c}{Stage~1 (Binary)} &")
print(r"\multicolumn{3}{c}{Stage~2 (Multi-class)} & \multirow{2}{*}{\makecell{Train\\time\\(s)}} \\")
print(r"\cmidrule(lr){2-4}\cmidrule(lr){5-7}")
print(r" & P & R & F\textsubscript{1} & P & R & F\textsubscript{1} & \\")
print(r"\midrule")
for r in results_algo:
    name = r["label"].replace(" (Proposed)","")
    prop = "(Proposed)" in r["label"]
    b1s  = "\\textbf{" if r["s1_f1"]==best_s1 else ""
    b1e  = "}" if r["s1_f1"]==best_s1 else ""
    b2s  = "\\textbf{" if r["s2_f1"]==best_s2 else ""
    b2e  = "}" if r["s2_f1"]==best_s2 else ""
    pfx  = "\\textit{" if prop else ""; sfx = "}" if prop else ""
    t    = r.get("train_time_sec","—")
    print(f"{pfx}{name}{sfx} & {r['s1_precision']:.2f} & {r['s1_recall']:.2f} & "
          f"{b1s}{r['s1_f1']:.2f}{b1e} & {r['s2_precision']:.2f} & {r['s2_recall']:.2f} & "
          f"{b2s}{r['s2_f1']:.2f}{b2e} & {t} \\\\")
print(r"\bottomrule")
print(r"\end{tabular}")
print(r"\end{table}")


## 6. Experiment 2 — Cascade vs. Flat Classifier

In [ ]:
from sklearn.metrics import classification_report as cr

print("Training flat 7-class CatBoost classifier...")
tr_flat = normalise_labels(preprocessing(pd.read_csv(TRAIN_CTX), with_context=True))
te_flat = normalise_labels(preprocessing(pd.read_csv(TEST_CTX),  with_context=True))
X_tr_flat = tr_flat[ALL_FEATURES_WITH_CTX]; y_tr_flat = tr_flat['output_2']
X_te_flat = te_flat[ALL_FEATURES_WITH_CTX]; y_te_flat = te_flat['output_2']
pool_tr_flat = Pool(X_tr_flat, label=y_tr_flat, cat_features=CATEGORY_FEATURES)
pool_te_flat = Pool(X_te_flat, label=y_te_flat, cat_features=CATEGORY_FEATURES)
task = "GPU" if USE_GPU else "CPU"
flat_model = CatBoostClassifier(iterations=5000, learning_rate=0.08, depth=8,
    l2_leaf_reg=7, task_type=task, logging_level='Silent')
with tqdm(total=5000, desc="  Flat 7-class training",
          bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} iter [{elapsed}]",
          colour="blue") as pb:
    flat_model.fit(pool_tr_flat, eval_set=pool_te_flat,
                   callbacks=[TqdmCatBoostCallback(5000, "  Flat model")])
y_flat_pred = np.array([p[0] if isinstance(p,(list,np.ndarray)) else p
                         for p in flat_model.predict(X_te_flat)])
flat_f1 = f1_score(y_te_flat, y_flat_pred, average='macro')
flat_pr = precision_score(y_te_flat, y_flat_pred, average='macro')
flat_rc = recall_score(y_te_flat,    y_flat_pred, average='macro')
print(f"  Flat 7-class — P={flat_pr:.3f}  R={flat_rc:.3f}  F1={flat_f1:.3f}")

y1_pred_cascade = res_cb['y1_pred']; y2_pred_cascade = res_cb['y2_pred']
y_cascade_full = []; j = 0
for p1 in y1_pred_cascade:
    if p1 == 'NORMAL': y_cascade_full.append('NORMAL')
    else: y_cascade_full.append(y2_pred_cascade[j]); j += 1
y_cascade_full = np.array(y_cascade_full)
te_full = normalise_labels(preprocessing(pd.read_csv(TEST_CTX), with_context=True))
y_true_7class = te_full['output_2'].values
cascade_f1_7 = f1_score(y_true_7class, y_cascade_full, average='macro')
cascade_pr_7 = precision_score(y_true_7class, y_cascade_full, average='macro')
cascade_rc_7 = recall_score(y_true_7class,    y_cascade_full, average='macro')
print(f"  Cascade — P={cascade_pr_7:.3f}  R={cascade_rc_7:.3f}  F1={cascade_f1_7:.3f}")

rep_flat = cr(y_true_7class, y_flat_pred,    output_dict=True, zero_division=0)
rep_casc = cr(y_true_7class, y_cascade_full, output_dict=True, zero_division=0)
classes_7 = ['BROKEN_AUTHENTICATION','BRUTE_FORCE','NORMAL',
             'SENSITIVE_DATA_EXPOSURE','SQL_INJECTION','WEB_SCAN','XSS']
rows = [{'Class':c, 'Flat F1':round(rep_flat[c]['f1-score'],3),
         'Cascade F1':round(rep_casc[c]['f1-score'],3),
         'Delta':round(rep_casc[c]['f1-score']-rep_flat[c]['f1-score'],3)}
        for c in classes_7 if c in rep_flat and c in rep_casc]
df_compare = pd.DataFrame(rows)
display(df_compare.style.bar(subset=['Delta'], color=['#f4a2a2','#a2f4a2']))

print("\n" + "="*70)
print("LATEX TABLE — Cascade vs Flat")
print("="*70)
print(r"\begin{table}[htbp]\centering")
print(r"\caption{Per-class F\textsubscript{1}: cascaded vs.~flat 7-class CatBoost.}")
print(r"\label{tab:cascade_vs_flat}\begin{tabular}{lccc}\toprule")
print(r"Class & Flat & Cascaded & $\Delta$ \\\midrule")
for _, row in df_compare.iterrows():
    s = "+" if row['Delta']>=0 else ""
    print(f"\\texttt{{{row['Class']}}} & {row['Flat F1']:.3f} & {row['Cascade F1']:.3f} & {s}{row['Delta']:.3f} \\\\")
print(f"Macro avg & {flat_f1:.3f} & {cascade_f1_7:.3f} & {cascade_f1_7-flat_f1:+.3f} \\\\")
print(r"\bottomrule\end{tabular}\end{table}")


## 7. Experiment 3 — Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
cm1 = confusion_matrix(res_cb['y1_test'], res_cb['y1_pred'], labels=['NORMAL','ATTACK'])
sns.heatmap(cm1, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['NORMAL','ATTACK'], yticklabels=['NORMAL','ATTACK'])
axes[0].set_title('Stage 1 — Binary Classifier\n(CatBoost + Context, N=30)', fontsize=13)
axes[0].set_ylabel('True Label'); axes[0].set_xlabel('Predicted Label')

labels_s2 = sorted(res_cb['y2_test'].unique())
cm2      = confusion_matrix(res_cb['y2_test'], res_cb['y2_pred'], labels=labels_s2)
cm2_norm = cm2.astype(float) / cm2.sum(axis=1, keepdims=True) * 100
annot    = np.array([[f"{v:.0f}\n({cm2[i,j]})" for j,v in enumerate(row)]
                     for i,row in enumerate(cm2_norm)])
sns.heatmap(cm2_norm, annot=annot, fmt='', cmap='YlOrRd', ax=axes[1],
            xticklabels=[l.replace('_','\n') for l in labels_s2],
            yticklabels=[l.replace('_','\n') for l in labels_s2])
axes[1].set_title('Stage 2 — Multi-class Attack Classifier\n(CatBoost + Context, N=30)', fontsize=13)
axes[1].set_ylabel('True Label'); axes[1].set_xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=200, bbox_inches='tight')
plt.show()
print("✓ Saved: confusion_matrices.png")


## 8. Experiment 4 — Extended Rule Engine Comparison

In [ ]:
RAW_DATASET = f"{BASE_PATH}/datasets/dataset.csv"
raw = normalise_labels(pd.read_csv(RAW_DATASET))
print(f"Raw dataset: {raw.shape}")
print(raw['output_2'].value_counts())


In [ ]:
RULE_KEYWORDS = {
    'SQL_INJECTION'          : ['sql', 'sqli', 'injection'],
    'XSS'                    : ['xss', 'cross-site'],
    'WEB_SCAN'               : ['scan', 'scanner', 'directory', 'T1083'],
    'BRUTE_FORCE'            : ['brute', 'brute_force', 'authentication_failed', 'T1110'],
    'BROKEN_AUTHENTICATION'  : ['authentication', 'T1212', 'credential'],
    'SENSITIVE_DATA_EXPOSURE': ['sensitive', 'T1190', 'error', 'exception', 'stack trace'],
}
def rule_identifies(row, keywords):
    text = ' '.join([str(row.get('_source.rule.description','')),
                      str(row.get('_source.rule.groups','')),
                      str(row.get('_source.rule.mitre.id',''))]).lower()
    return any(kw.lower() in text for kw in keywords)

rule_results = []
for cls, kws in tqdm(RULE_KEYWORDS.items(), desc="Rule engine analysis"):
    subset = raw[raw['output_2'] == cls]
    if len(subset) == 0: continue
    identified = subset.apply(lambda r: rule_identifies(r, kws), axis=1).sum()
    rule_results.append({'Class':cls, 'Total Events':len(subset),
                          'Rule-identified':identified,
                          'Rule Coverage %':round(100*identified/len(subset),1)})

df_rule = pd.DataFrame(rule_results)
full_test = normalise_labels(preprocessing(pd.read_csv(TEST_CTX), with_context=True))
X_full = full_test[ALL_FEATURES_WITH_CTX]; y_true = full_test['output_2'].values
pred1 = res_cb['m1'].predict(X_full); y_cascade_pred = list(pred1)
attack_idx = [i for i,p in enumerate(pred1) if p=='ATTACK']
pred2 = res_cb['m2'].predict(X_full.iloc[attack_idx])
pred2 = [p[0] if isinstance(p,(list,np.ndarray)) else p for p in pred2]
for k,i in enumerate(attack_idx): y_cascade_pred[i] = pred2[k]
for i,row in df_rule.iterrows():
    cls = row['Class']
    tp  = sum(1 for t,p in zip(y_true, y_cascade_pred) if t==cls and p==cls)
    tot = sum(1 for t in y_true if t==cls)
    df_rule.at[i,'AI Detected']   = tp
    df_rule.at[i,'AI Coverage %'] = round(100*tp/tot,1) if tot>0 else 0
    df_rule.at[i,'AI F1']         = round(f1_score(y_true, y_cascade_pred, labels=[cls], average='macro'),3)
display(df_rule)
print("\n" + "="*70)
print("LATEX TABLE — Extended rule engine comparison")
print("="*70)
print(r"\begin{table}[htbp]\centering")
print(r"\caption{Wazuh rule engine vs.\ \textsc{Smart-SIEM} coverage across all six attack classes.}")
print(r"\label{tab:wazuh_vs_ai_full}\begin{tabular}{lrrrrr}\toprule")
print(r"Class & Total & Rule & Rule\% & AI & AI\% \\\midrule")
for _,row in df_rule.iterrows():
    print(f"\\texttt{{{row['Class']}}} & {int(row['Total Events']):,} & "
          f"{int(row['Rule-identified']):,} & {row['Rule Coverage %']:.1f} & "
          f"{int(row['AI Detected']):,} & {row['AI Coverage %']:.1f} \\\\")
print(r"\bottomrule\end{tabular}\end{table}")


## 9. Experiment 5 — Exact Ablation Study

In [ ]:
SKIP_ABLATION = False  # Set True + fill MANUAL_ABLATION to skip retraining
N_VALUES = [3, 5, 7, 10, 12, 15, 20, 25, 30, 35]

if SKIP_ABLATION:
    MANUAL_ABLATION = {3:(0.80,0.70),5:(0.82,0.74),7:(0.84,0.76),10:(0.87,0.81),
                       12:(0.89,0.83),15:(0.91,0.86),20:(0.93,0.88),25:(0.94,0.89),
                       30:(0.95,0.90),35:(0.95,0.90)}
    ablation_results = [{'N':n,'s1_f1':v[0],'s2_f1':v[1]} for n,v in MANUAL_ABLATION.items()]
    print("Using manual values")
else:
    ablation_results = []
    task = "GPU" if USE_GPU else "CPU"
    for N in tqdm(N_VALUES, desc="Ablation (N values)"):
        tr_path = f"{BASE_PATH}/datasets/training/final_training_dataset_with_history_{N}_SMOTENC.csv"
        te_path = f"{BASE_PATH}/datasets/testing/final_testing_dataset_with_history_{N}.csv"
        if not os.path.exists(tr_path) or not os.path.exists(te_path):
            print(f"  N={N}: files not found, skipping"); continue
        try:
            Xtr,y1tr,X2tr,y2tr,Xte,y1te,X2te,y2te = load_dataset(tr_path, te_path)
            res = train_catboost_cascade(
                Xtr,y1tr,X2tr,y2tr,Xte,y1te,X2te,y2te,
                params1=dict(iterations=2631,learning_rate=0.1,depth=10,l2_leaf_reg=10),
                params2=dict(iterations=6405,learning_rate=0.08,depth=8,l2_leaf_reg=7),
                label=f"N={N}", tune=False)
            ablation_results.append({'N':N,'s1_f1':res['s1_f1'],'s2_f1':res['s2_f1']})
            print(f"  N={N}: S1={res['s1_f1']:.3f}  S2={res['s2_f1']:.3f}")
        except Exception as e: print(f"  N={N}: ERROR {e}")

df_abl = pd.DataFrame(ablation_results)
display(df_abl)
with open('ablation_results.json','w') as f: json.dump(ablation_results,f,indent=2)
print("✓ Saved: ablation_results.json")


In [ ]:
fig, ax = plt.subplots(figsize=(9,5))
ax.plot(df_abl['N'], df_abl['s1_f1'], 'b-o', label='Stage 1 (Binary)',     linewidth=2)
ax.plot(df_abl['N'], df_abl['s2_f1'], 'g-s', label='Stage 2 (Multi-class)', linewidth=2)
ax.axhline(1.0, color='red', linestyle='--', label='Upper bound', linewidth=1.5)
ax.set_xlabel('Context window size $N$', fontsize=12)
ax.set_ylabel('Macro F₁-score', fontsize=12)
ax.set_ylim(0.65, 1.02); ax.set_xticks(N_VALUES)
ax.legend(fontsize=11); ax.grid(True, alpha=0.4)
plt.title('Macro F₁-score vs Context Window Size N', fontsize=13)
plt.tight_layout()
plt.savefig('ablation_curve.png', dpi=200, bbox_inches='tight')
plt.show()
print("✓ Saved: ablation_curve.png")
print("\n--- EXACT VALUES FOR TIKZ ---")
s1 = " ".join([f"({r['N']},{r['s1_f1']:.3f})" for _,r in df_abl.iterrows()])
s2 = " ".join([f"({r['N']},{r['s2_f1']:.3f})" for _,r in df_abl.iterrows()])
print(f"% Stage 1\n\\addplot coordinates {{{s1}}};")
print(f"% Stage 2\n\\addplot coordinates {{{s2}}};")


## 10. Experiment 6 — Self-Adaptive Retraining Simulation

In [ ]:
raw_full = normalise_labels(pd.read_csv(RAW_DATASET))
raw_full = preprocessing(raw_full, with_context=False)
if 'output_1' not in raw_full.columns:
    raw_full['output_1'] = raw_full['output_2'].apply(lambda x: 'NORMAL' if x=='NORMAL' else 'ATTACK')
n  = len(raw_full); p1 = int(n*0.60); p2 = int(n*0.80)
phase1 = raw_full.iloc[:p1]; phase2 = raw_full.iloc[p1:p2]; phase3 = raw_full.iloc[p2:]
print(f"Phase 1: {len(phase1):,}  Phase 2: {len(phase2):,}  Phase 3: {len(phase3):,}")


In [ ]:
feats = ALL_FEATURES_NO_CTX
X_p1=phase1[feats]; y_p1=phase1['output_1']
X_p2=phase2[feats]; y_p2=phase2['output_1']
X_p3=phase3[feats]; y_p3=phase3['output_1']
task = "GPU" if USE_GPU else "CPU"
initial_pool = Pool(X_p1, label=y_p1, cat_features=CATEGORY_FEATURES)
initial_model = CatBoostClassifier(iterations=1000,learning_rate=0.1,depth=8,
                                    task_type=task,logging_level='Silent')
with tqdm(total=1000,desc="  Initial model training",colour="blue",
          bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} iter [{elapsed}]") as pb:
    initial_model.fit(initial_pool, callbacks=[TqdmCatBoostCallback(1000,"  Initial")])
acc_on_p1 = f1_score(y_p1, initial_model.predict(X_p1), average='macro')
acc_on_p2 = f1_score(y_p2, initial_model.predict(X_p2), average='macro')
acc_on_p3 = f1_score(y_p3, initial_model.predict(X_p3), average='macro')
print(f"Initial model: P1={acc_on_p1:.3f}  P2={acc_on_p2:.3f}  P3={acc_on_p3:.3f}")
THRESHOLD = 0.90
print(f"Retrain triggered (P2 < {THRESHOLD:.0%}): {acc_on_p2 < THRESHOLD}")


In [ ]:
retrain_pool = Pool(X_p2, label=y_p2, cat_features=CATEGORY_FEATURES)
retrained_model = CatBoostClassifier(iterations=1000,learning_rate=0.1,depth=8,
                                      task_type=task,logging_level='Silent')
with tqdm(total=1000,desc="  Retrained model",colour="blue",
          bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} iter [{elapsed}]") as pb:
    retrained_model.fit(retrain_pool, callbacks=[TqdmCatBoostCallback(1000,"  Retrain")])
acc_retrained_p2 = f1_score(y_p2, retrained_model.predict(X_p2), average='macro')
acc_retrained_p3 = f1_score(y_p3, retrained_model.predict(X_p3), average='macro')
print(f"Retrained model: P2={acc_retrained_p2:.3f}  P3={acc_retrained_p3:.3f}")

df_retrain = pd.DataFrame([
    {'Phase':'Phase 1 (Training)','Initial Model':f'{acc_on_p1:.3f}','Retrained Model':'—'},
    {'Phase':'Phase 2 (Drift)',    'Initial Model':f'{acc_on_p2:.3f}','Retrained Model':f'{acc_retrained_p2:.3f}'},
    {'Phase':'Phase 3 (Holdout)',  'Initial Model':f'{acc_on_p3:.3f}','Retrained Model':f'{acc_retrained_p3:.3f}'},
])
display(df_retrain)

fig,ax = plt.subplots(figsize=(8,4))
phases=['Phase 1\n(Training)','Phase 2\n(Drift)','Phase 3\n(Holdout)']
x=np.arange(3); w=0.35
ax.bar(x-w/2,[acc_on_p1,acc_on_p2,acc_on_p3],      w,label='Initial Model',  color='#5494d8',alpha=0.85)
ax.bar(x+w/2,[acc_on_p1,acc_retrained_p2,acc_retrained_p3],w,label='Retrained Model',color='#4caf50',alpha=0.85)
ax.axhline(THRESHOLD,color='red',linestyle='--',label=f'Threshold ({THRESHOLD:.0%})')
ax.set_xticks(x); ax.set_xticklabels(phases); ax.set_ylabel('Macro F₁'); ax.set_ylim(0,1.05)
ax.legend(); ax.grid(axis='y',alpha=0.4)
ax.set_title('Self-Adaptive Retraining: Before and After',fontsize=12)
plt.tight_layout()
plt.savefig('retraining_simulation.png',dpi=200,bbox_inches='tight')
plt.show()
print("✓ Saved: retraining_simulation.png")
print("\n" + "="*60)
print("LATEX TABLE — Retraining simulation")
print("="*60)
print(r"\begin{table}[htbp]\centering")
print(r"\caption{Self-adaptive retraining simulation.}\label{tab:retrain_sim}")
print(r"\begin{tabular}{lcc}\toprule")
print(r"Data Segment & Initial Model & Retrained Model \\\midrule")
print(f"Phase 1 (training)      & {acc_on_p1:.3f} & --- \\\\")
print(f"Phase 2 (drift)         & {acc_on_p2:.3f} & {acc_retrained_p2:.3f} \\\\")
print(f"Phase 3 (holdout)       & {acc_on_p3:.3f} & {acc_retrained_p3:.3f} \\\\")
print(r"\bottomrule\end{tabular}\end{table}")


## 11. Full Results Summary

In [ ]:
print("="*70)
print("COMPLETE RESULTS SUMMARY")
print("="*70)
print("\n[Algorithm Comparison]")
for r in results_algo:
    print(f"  {r['label']:<35} S1={r['s1_f1']:.3f}  S2={r['s2_f1']:.3f}  {r.get('train_time_sec','—')}s")
print("\n[Cascade vs Flat]")
print(f"  Flat       : {flat_f1:.3f}")
print(f"  Cascaded   : {cascade_f1_7:.3f}  ({cascade_f1_7-flat_f1:+.3f})")
print("\n[Ablation]")
for _,row in df_abl.iterrows():
    print(f"  N={int(row['N']):2d}: S1={row['s1_f1']:.3f}  S2={row['s2_f1']:.3f}")
print("\n[Retraining]")
print(f"  Initial on drift: {acc_on_p2:.3f}  |  Retrained on holdout: {acc_retrained_p3:.3f}")
print("\n[Output files]")
for fname in ['confusion_matrices.png','ablation_curve.png',
              'retraining_simulation.png','ablation_results.json',
              'diagram_A_algorithm_comparison.png',
              'diagram_B_radar_chart.png','diagram_C_efficiency_accuracy.png']:
    status = "✓" if os.path.exists(fname) else "✗ MISSING"
    print(f"  {status}  {fname}")


---
## ✅ After running: what to do

1. **Algorithm comparison table** — printed at the end of Cell 13 → paste into paper §8
2. **Cascade vs flat table** — printed in Cell 15 → paste into paper §8
3. **Confusion matrices** → `confusion_matrices.png` → include as figure in paper §8.2
4. **Ablation TikZ coordinates** — printed in Cell 23 → replace `\addplot coordinates` in the LaTeX
5. **Retraining table** — printed in Cell 27 → paste into paper §8
6. **Diagrams A/B/C** → `.png` files → include as figures in paper §5
7. Send me the filled-in numbers and I will update the full paper automatically
